In [0]:
import pandas as pd
import json

# Load IDP results
df = spark.table("default.ghana_idp_results").toPandas()
# Load original for extra fields
df_orig = spark.table("default.ghana_facilities_clean").toPandas()

# Merge
df_full = df.merge(
    df_orig[['unique_id', 'numberDoctors', 'capacity', 'operatorTypeId', 
             'affiliationTypeIds', 'specialties_list', 'source_url']],
    on='unique_id', how='left'
)

print("Loaded:", len(df_full), "facilities")

# Define a medical desert score (0-100, higher = more underserved)
def desert_score(row):
    score = 0
    if not row['has_procedures']:   score += 30
    if not row['has_equipment']:    score += 35
    if not row['has_capabilities']: score += 20
    caps = row['idp_capabilities'].lower()
    if 'icu' not in caps and 'intensive' not in caps: score += 5
    if 'emergency' not in caps:                        score += 5
    if 'surgery' not in caps and 'surgical' not in caps: score += 5
    return score

df_full['desert_score'] = df_full.apply(desert_score, axis=1)

# Flag medical deserts (score >= 60)
df_full['is_medical_desert'] = df_full['desert_score'] >= 60

print("Medical deserts identified:", df_full['is_medical_desert'].sum())
print("\nDesert score distribution:")
print(df_full['desert_score'].value_counts().sort_index())

Loaded: 1067 facilities
Medical deserts identified: 758

Desert score distribution:
desert_score
5        8
10      31
15      43
40       8
45      66
50     153
70       9
75      25
80     644
100     80
Name: count, dtype: int64


In [0]:
# Regional breakdown of medical deserts
regional = df_full.groupby('address_stateOrRegion').agg(
    total_facilities=('unique_id', 'count'),
    medical_deserts=('is_medical_desert', 'sum'),
    avg_desert_score=('desert_score', 'mean'),
    has_procedures=('has_procedures', 'sum'),
    has_equipment=('has_equipment', 'sum'),
    has_capabilities=('has_capabilities', 'sum'),
).reset_index()

regional['desert_pct'] = (regional['medical_deserts'] / regional['total_facilities'] * 100).round(1)
regional = regional.sort_values('desert_pct', ascending=False)

print("REGIONAL MEDICAL DESERT ANALYSIS")
print("="*60)
for _, row in regional.iterrows():
    region = str(row['address_stateOrRegion'])[:30] if row['address_stateOrRegion'] else 'Unknown'
    print(f"{region:<30} {row['medical_deserts']:>3}/{row['total_facilities']:>3} deserts ({row['desert_pct']}%)")

# Save full analyzed dataset
spark.createDataFrame(df_full).write.mode("overwrite").saveAsTable("default.ghana_gap_analysis")
print("\nSaved to: default.ghana_gap_analysis")

REGIONAL MEDICAL DESERT ANALYSIS
ASHANTI                          6/  6 deserts (100.0%)
Northern                         4/  4 deserts (100.0%)
KEEA                             1/  1 deserts (100.0%)
Oti                              1/  1 deserts (100.0%)
SH                               1/  1 deserts (100.0%)
ASHANTI REGION                   1/  1 deserts (100.0%)
Savannah                         1/  1 deserts (100.0%)
Sissala West District            1/  1 deserts (100.0%)
Ejisu Municipal                  1/  1 deserts (100.0%)
Eastern Region                   2/  2 deserts (100.0%)
Eastern                          2/  2 deserts (100.0%)
Takoradi                         1/  1 deserts (100.0%)
Dormaa East                      1/  1 deserts (100.0%)
Central Tongu District           1/  1 deserts (100.0%)
Central Region                  10/ 10 deserts (100.0%)
Techiman Municipal               1/  1 deserts (100.0%)
Central                          5/  5 deserts (100.0%)
Tema West Munic

In [0]:
# Detect suspicious/anomalous facility claims
def detect_anomalies(row):
    anomalies = []
    caps = str(row['idp_capabilities']).lower()
    procs = str(row['idp_procedures']).lower()
    equip = str(row['idp_equipment']).lower()
    
    # Claims advanced care but no equipment
    if ('icu' in caps or 'surgery' in caps) and not row['has_equipment']:
        anomalies.append("Claims advanced care but no equipment listed")
    
    # Claims to be a hospital but no capabilities
    if row['facilityTypeId'] == 'hospital' and not row['has_capabilities']:
        anomalies.append("Listed as hospital but no capabilities found")
    
    # Claims surgery but no surgical equipment
    if 'surgery' in procs and 'theater' not in equip and 'theatre' not in equip:
        anomalies.append("Claims surgery but no theater mentioned")
    
    # Has ICU claim but no ventilator or monitor
    if 'icu' in caps and 'ventilator' not in equip and 'monitor' not in equip:
        anomalies.append("Claims ICU but no ICU equipment found")

    # No data at all
    if not row['has_procedures'] and not row['has_equipment'] and not row['has_capabilities']:
        anomalies.append("No medical data found - needs field verification")

    return anomalies

df_full['anomalies'] = df_full.apply(detect_anomalies, axis=1)
df_full['anomaly_count'] = df_full['anomalies'].apply(len)
df_full['has_anomaly'] = df_full['anomaly_count'] > 0

print("ANOMALY DETECTION RESULTS")
print("="*60)
print(f"Total facilities with anomalies: {df_full['has_anomaly'].sum()}")
print(f"\nAnomaly breakdown:")
all_anomalies = [a for anomalies in df_full['anomalies'] for a in anomalies]
from collections import Counter
for anomaly, count in Counter(all_anomalies).most_common():
    print(f"  {count:>4}x  {anomaly}")

print(f"\nTop 5 most suspicious facilities:")
suspicious = df_full[df_full['anomaly_count'] > 0].sort_values('anomaly_count', ascending=False)
for _, row in suspicious.head(5).iterrows():
    print(f"\n  {row['name']} ({row['address_city']})")
    for a in row['anomalies']:
        print(f"    ⚠️  {a}")

# Drop and recreate with new schema
df_full['anomalies'] = df_full['anomalies'].apply(json.dumps)
df_full['anomaly_count'] = df_full['anomaly_count'].astype(int)
df_full['has_anomaly'] = df_full['has_anomaly'].astype(bool)

spark.sql("DROP TABLE IF EXISTS default.ghana_gap_analysis")
spark.createDataFrame(df_full).write.mode("overwrite").saveAsTable("default.ghana_gap_analysis")
print("Saved to: default.ghana_gap_analysis")

ANOMALY DETECTION RESULTS
Total facilities with anomalies: 169

Anomaly breakdown:
    80x  No medical data found - needs field verification
    55x  Claims surgery but no theater mentioned
    46x  Claims advanced care but no equipment listed
     5x  Claims ICU but no ICU equipment found
     3x  Listed as hospital but no capabilities found

Top 5 most suspicious facilities:

  Saint John of God Hospital, Duayaw Nkwanta (Duayaw Nkwanta)
    ⚠️  Claims advanced care but no equipment listed
    ⚠️  Claims surgery but no theater mentioned
    ⚠️  Claims ICU but no ICU equipment found

  Accra Specialist Eye Hospital (Accra)
    ⚠️  Listed as hospital but no capabilities found
    ⚠️  No medical data found - needs field verification

  Sam-j Hospital (Accra)
    ⚠️  Claims surgery but no theater mentioned
    ⚠️  Claims ICU but no ICU equipment found

  Imperial Nursing And Home-care Services (null)
    ⚠️  Claims advanced care but no equipment listed
    ⚠️  Claims surgery but no theate